# Tạo file đầu vào thống nhất cho mô hình dự báo CPI

Notebook này tạo lại dataset đầu vào dựa trên schema các file dữ liệu đã cung cấp.

Điểm sửa chính:
- Target `cpi_mom` lấy từ cột `inflation`, không lấy cột `cpi` index.
- Giữ thêm `cpi_index` để tham khảo.
- Không forward-fill target.
- Đọc file đúng từ `data/raw` hoặc thư mục hiện tại.
- Parse ngày theo nhiều format, ưu tiên `dayfirst=True`.
- Aggregate dữ liệu ngày về tháng bằng `raw_date`.
- Tạo feature đầu vào dạng lag/rolling đã shift để hạn chế leakage.
- Lưu dataset processed dùng cho VAR, SARIMAX, VECM, ML.

## 1. Thiết lập

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

SCRIPT_DIR = os.getcwd()
RAW_DIR = os.path.join(SCRIPT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(SCRIPT_DIR, "data", "processed")
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "outputs", "unified_input")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Script dir:", SCRIPT_DIR)
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

Script dir: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test
Raw dir: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test\data\raw
Processed dir: d:\Nam4-HK1\KLTN\Vietnam_economic_lakehouse\model_test\data\processed


## 2. Khai báo schema dữ liệu

In [11]:
DATA_CONFIG = {
    # CPI và biến vĩ mô
    "cpi_mom.csv": {"type": "cpi", "date_col": "date", "value_cols": ["cpi", "inflation"]},
    "core_inflation_rate.csv": {"type": "macro", "date_col": "date", "value_cols": ["value"], "rename": "core_inflation_rate"},
    #"cpi_base_year.csv": {"type": "macro", "date_col": "date", "value_cols": ["base_2010"], "rename": "cpi_base_year"},
    "interest_rate.csv": {"type": "interest_rate", "date_col": "date", "value_cols": ["interest_rate"]},
    "ppi_qoq.csv": {"type": "macro", "date_col": "date", "value_cols": ["value"], "rename": "ppi_qoq"},
    "m2.csv": {"type": "macro", "date_col": "date", "value_cols": ["value"], "rename": "m2"},

    # Năng lượng/hàng hóa
    "brent.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "brent"},
    "wti.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "wti"},
    "gasoline.csv": {"type": "gasoline", "date_col": "date", "value_cols": ["price"]},
    "gasoline_world.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "gasoline_world"},
    "natural_gas.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "natural_gas"},
    "gold.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "gold"},
    "silver.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "silver"},

    # Thị trường Việt Nam
    "VNINDEX.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "VNINDEX"},
    "VN30.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "VN30"},
    "HNX.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "HNX"},
    "HNX30.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "HNX30"},
    "UPCOM.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "UPCOM"},

    # Thị trường quốc tế
    "NASDAQ.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "NASDAQ"},
    "S&P500.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "S&P500"},
    "DAX.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "DAX"},
    "DOWJONES.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "DOWJONES"},
    "NIKKEI225.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "NIKKEI225"},
    "HANGSENG.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "HANGSENG"},

    # Tỷ giá
    "USDVND.csv": {"type": "ohlc", "date_col": "date", "value_cols": ["close"], "rename": "USDVND"},
}

print("", len(DATA_CONFIG))

 24


## 3. Hàm 

In [12]:
def find_file(filename):
    raw_path = os.path.join(RAW_DIR, filename)
    cwd_path = os.path.join(SCRIPT_DIR, filename)
    if os.path.exists(raw_path):
        return raw_path
    if os.path.exists(cwd_path):
        return cwd_path
    return None


def parse_date_series(series):
    s = series.astype(str).str.strip()
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    miss = dt.isna()
    if miss.any():
        dt2 = pd.to_datetime(s[miss], errors="coerce", dayfirst=False)
        dt.loc[miss] = dt2
    return dt


def load_raw_file(filename, cfg):
    path = find_file(filename)
    if path is None:
        return None, f"Không tìm thấy file {filename}"

    df = pd.read_csv(path)
    date_col = cfg.get("date_col", "date")
    if date_col not in df.columns:
        return None, f"Không có cột ngày {date_col}"

    df = df.copy()
    df["raw_date"] = parse_date_series(df[date_col])
    df = df.dropna(subset=["raw_date"])
    df["date"] = df["raw_date"].dt.to_period("M").dt.to_timestamp()
    return df, None


def safe_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

## 4. Load tất cả file

In [13]:
raw_data = {}
failed = []

for filename, cfg in DATA_CONFIG.items():
    df, err = load_raw_file(filename, cfg)
    if err is not None:
        failed.append((filename, err))
        print("FAIL", filename, "-", err)
    else:
        raw_data[filename] = df
        print("OK", filename, df.shape, df["date"].min().date(), "đến", df["date"].max().date())

print("Loaded:", len(raw_data))
print("Failed:", len(failed))

OK cpi_mom.csv (360, 8) 1995-01-01 đến 2024-01-01
OK core_inflation_rate.csv (131, 6) 2015-01-01 đến 2026-01-01
OK interest_rate.csv (765, 8) 2014-01-01 đến 2026-12-01
OK ppi_qoq.csv (40, 6) 2016-01-01 đến 2025-01-01
OK m2.csv (195, 7) 2009-01-01 đến 2026-01-01
OK brent.csv (4658, 14) 2007-07-01 đến 2026-04-01
OK wti.csv (9260, 14) 1990-01-01 đến 2026-12-01
OK gasoline.csv (1107, 9) 2018-08-01 đến 2026-04-01
OK gasoline_world.csv (6395, 14) 2000-11-01 đến 2026-04-01
OK natural_gas.csv (6437, 14) 2000-08-01 đến 2026-04-01
OK gold.csv (6431, 14) 2000-08-01 đến 2026-04-01
OK silver.csv (6433, 14) 2000-08-01 đến 2026-04-01
OK VNINDEX.csv (6240, 13) 2000-07-01 đến 2026-03-01
OK VN30.csv (4290, 13) 2009-01-01 đến 2026-12-01
OK HNX.csv (5071, 13) 2005-07-01 đến 2026-03-01
OK HNX30.csv (159, 13) 2024-01-01 đến 2025-12-01
OK UPCOM.csv (4177, 13) 2009-06-01 đến 2026-03-01
OK NASDAQ.csv (13916, 14) 1971-01-01 đến 2026-12-01
OK S&P500.csv (24690, 14) 1927-12-01 đến 2026-04-01
OK DAX.csv (9683, 14)

## 5. Aggregate từng nguồn về tháng

In [19]:
def aggregate_cpi(df):
    df = df.sort_values("raw_date").drop_duplicates(subset=["date"], keep="last")
    df = safe_numeric(df, ["cpi", "inflation"])

    if "inflation" not in df.columns:
        raise ValueError("cpi.csv thiếu cột inflation")

    out = df[["date", "cpi", "inflation"]].copy()
    return out

def aggregate_macro(df, out_name, value_col="value"):
    df = safe_numeric(df, [value_col])
    out = (
        df.sort_values("raw_date")
        .dropna(subset=[value_col])
        .drop_duplicates(subset=["date"], keep="last")
        [["date", value_col]]
        .rename(columns={value_col: out_name})
        .reset_index(drop=True)
    )
    return out


def aggregate_ohlc(df, out_name):
    df = safe_numeric(df, ["close", "open", "high", "low", "volume"])
    agg_dict = {}
    if "close" in df.columns:
        agg_dict[f"{out_name}_last"] = ("close", "last")
        agg_dict[f"{out_name}_mean"] = ("close", "mean")
    if "volume" in df.columns:
        agg_dict[f"{out_name}_volume"] = ("volume", "sum")

    out = (
        df.sort_values("raw_date")
        .groupby("date")
        .agg(**agg_dict)
        .reset_index()
    )
    return out


def aggregate_gasoline(df):
    df = df.copy()
    if "product" in df.columns:
        ron95 = df[df["product"].astype(str).str.upper().str.contains("RON 95", regex=False)].copy()
        if len(ron95) > 0:
            df = ron95
    df = safe_numeric(df, ["price", "change"])
    out = (
        df.sort_values("raw_date")
        .groupby("date")
        .agg(
            gasoline_price_last=("price", "last"),
            gasoline_price_mean=("price", "mean"),
            gasoline_change_last=("change", "last")
        )
        .reset_index()
    )
    return out


def aggregate_interest_rate(df):
    df = df.copy()
    df = safe_numeric(df, ["interest_rate"])
    symbol_mapping = {
        "ON": "interest_rate_ON",
        "1W": "interest_rate_1W",
        "2W": "interest_rate_2W",
        "1M": "interest_rate_1M",
        "3M": "interest_rate_3M",
        "6M": "interest_rate_6M",
        "9M": "interest_rate_9M",
    }
    df["symbol"] = df["symbol"].map(symbol_mapping).fillna(df["symbol"])
    month = (
        df.sort_values("raw_date")
        .groupby(["date", "symbol"])
        .agg(interest_rate=("interest_rate", "last"))
        .reset_index()
    )
    out = month.pivot(index="date", columns="symbol", values="interest_rate").reset_index()
    out.columns.name = None
    return out


monthly_data = {}

for filename, df in raw_data.items():
    cfg = DATA_CONFIG[filename]
    try:
        if cfg["type"] == "cpi":
            monthly_data[filename] = aggregate_cpi(df)
        elif cfg["type"] == "macro":
            monthly_data[filename] = aggregate_macro(df, cfg["rename"], cfg["value_cols"][0])
        elif cfg["type"] == "ohlc":
            monthly_data[filename] = aggregate_ohlc(df, cfg["rename"])
        elif cfg["type"] == "gasoline":
            monthly_data[filename] = aggregate_gasoline(df)
        elif cfg["type"] == "interest_rate":
            monthly_data[filename] = aggregate_interest_rate(df)
        print("OK monthly", filename, monthly_data[filename].shape)
    except Exception as e:
        print("FAIL monthly", filename, e)

OK monthly cpi_mom.csv (30, 3)
OK monthly core_inflation_rate.csv (12, 2)
OK monthly interest_rate.csv (23, 9)
OK monthly ppi_qoq.csv (10, 2)
OK monthly m2.csv (18, 2)
OK monthly brent.csv (226, 4)
OK monthly wti.csv (444, 4)
OK monthly gasoline.csv (93, 4)
OK monthly gasoline_world.csv (306, 4)
OK monthly natural_gas.csv (309, 4)
OK monthly gold.csv (309, 4)
OK monthly silver.csv (309, 4)
OK monthly VNINDEX.csv (309, 4)
OK monthly VN30.csv (216, 4)
OK monthly HNX.csv (249, 4)
OK monthly HNX30.csv (24, 4)
OK monthly UPCOM.csv (202, 4)
OK monthly NASDAQ.csv (672, 4)
OK monthly S&P500.csv (1181, 4)
OK monthly DAX.csv (461, 4)
OK monthly DOWJONES.csv (420, 4)
OK monthly NIKKEI225.csv (744, 4)
OK monthly HANGSENG.csv (473, 4)
OK monthly USDVND.csv (396, 4)


## 6. Merge dataset đầu vào

In [21]:
if "cpi_mom.csv" not in monthly_data:
    raise ValueError("Không có CPI target để làm base")

base = monthly_data["cpi_mom.csv"].copy()
base = base.sort_values("date").reset_index(drop=True)

merged = base.copy()

for filename, table in monthly_data.items():
    if filename == "cpi_mom.csv":
        continue
    merged = merged.merge(table, on="date", how="left")

merged = merged.sort_values("date").reset_index(drop=True)

print("Merged:", merged.shape)
print("Date range:", merged["date"].min().date(), "đến", merged["date"].max().date())
print("Target describe:")
print(merged["cpi"].describe().to_string())

Merged: (30, 71)
Date range: 1995-01-01 đến 2024-01-01
Target describe:
count     30.000000
mean     100.542667
std        0.713539
min       99.320000
25%      100.105000
50%      100.400000
75%      100.800000
max      102.900000


In [22]:
merged.head(10)

,date,cpi,inflation,core_inflation_rate,Unknown,interest_rate_1M,interest_rate_1W,interest_rate_2W,interest_rate_3M,interest_rate_6M,...,DOWJONES_volume,NIKKEI225_last,NIKKEI225_mean,NIKKEI225_volume,HANGSENG_last,HANGSENG_mean,HANGSENG_volume,USDVND_last,USDVND_mean,USDVND_volume
0,1995-01-01,100.3,0.199800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,649850000,18649.820312,18092.322852,0,7342.700195,7454.470020,0,11039.0,11037.761905,0.0
1,1996-01-01,101.0,0.099108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,794310000,20812.740234,20784.086426,0,11359.700195,10717.381880,0,11010.0,11013.809524,0.0
2,1997-01-01,101.0,0.697906,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,991560000,18330.009766,18119.390067,0,13321.799805,13456.781916,0,11177.0,11362.409091,0.0
3,1998-01-01,100.8,0.699301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1445150000,16628.470703,15951.641992,0,9252.400391,9248.061117,0,12288.0,12650.000000,0.0
4,1999-01-01,100.5,0.099602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2010720000,14499.250000,15253.329939,0,9506.900391,10084.596484,0,13877.0,13906.227273,0.0
5,2000-01-01,100.1,-0.792864,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3988090000,19539.699219,18362.129511,0,15532.339844,15702.185268,0,14050.0,14090.714286,0.0
6,2001-01-01,101.0,0.798403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4812050000,13843.549805,13298.716553,0,16102.349609,15501.087942,0,14543.0,14615.700000,0.0
7,2002-01-01,100.3,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5284820000,9997.799805,10142.069987,179200000,10725.299805,11111.749467,6174800000,15119.0,15160.727273,0.0
8,2003-01-01,100.8,0.198807,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5095170000,8339.940430,8864.894741,1398800000,9258.950195,9574.506603,3885500000,15426.0,15447.000000,0.0
9,2004-01-01,100.6,0.399202,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4734090000,10783.610352,11024.613592,1696400000,13289.370117,13326.113076,7420500000,15693.0,15698.863636,0.0


In [23]:
merged.tail(10)

,date,cpi,inflation,core_inflation_rate,Unknown,interest_rate_1M,interest_rate_1W,interest_rate_2W,interest_rate_3M,interest_rate_6M,...,DOWJONES_volume,NIKKEI225_last,NIKKEI225_mean,NIKKEI225_volume,HANGSENG_last,HANGSENG_mean,HANGSENG_volume,USDVND_last,USDVND_mean,USDVND_volume
20,2015-01-01,100.02,-0.049965,1.69,NaN,4.12,4.02,4.22,4.66,4.60,...,2241550000,17674.390625,18001.795387,3057200000,24507.050781,24203.401507,39442000000,21267.0,21537.772727,0.0
21,2016-01-01,100.23,-0.248806,1.87,0.0,3.52,3.28,3.50,4.73,5.12,...,2760070000,17518.300781,16968.833097,3736400000,19683.109375,19733.578613,40565300000,22198.0,22341.043478,0.0
22,2017-01-01,100.21,0.079896,1.29,0.0,2.26,1.52,1.29,4.60,5.23,...,6834930000,19041.339844,19568.838821,2531600000,23360.779297,22814.503084,25822900000,22590.0,22623.904762,0.0
23,2018-01-01,99.75,0.040116,1.70,NaN,4.74,4.67,4.69,4.86,5.00,...,8368980000,23098.289062,23343.937988,1674100000,32887.269531,31831.893200,65423000000,22700.0,22801.047619,0.0
24,2019-01-01,101.40,0.435816,2.78,NaN,2.36,2.02,2.12,4.56,4.63,...,6916750000,20773.490234,20995.064063,1284400000,27942.470703,26711.562944,38581000000,23198.0,23204.043478,340.0
25,2020-01-01,100.10,0.110011,0.99,NaN,0.62,0.14,0.20,1.20,2.76,...,7225760000,23205.179688,23171.711035,1263800000,26312.630859,28214.347461,32702100000,23222.0,23206.913043,440.0
26,2021-01-01,99.82,-0.498405,0.67,NaN,1.16,0.75,0.88,2.23,2.56,...,8396800000,27663.390625,28627.061257,1548700000,28283.710938,28662.920410,73416300000,23048.0,23003.000000,420.0
27,2022-01-01,99.99,-0.398446,4.99,NaN,11.25,7.28,6.01,9.43,9.90,...,8873520000,27001.980469,27460.444158,1614500000,23802.259766,23974.514695,41535600000,22645.0,22984.681818,530.0
28,2023-01-01,100.12,-0.129676,2.98,NaN,2.08,1.49,1.68,3.69,5.22,...,6566450000,27327.109375,28347.210286,1651200000,21842.330078,21569.808485,46990600000,23445.0,23593.285714,20.0
29,2024-01-01,100.29,0.159792,2.85,NaN,4.81,4.16,4.04,4.95,5.10,...,7345230000,36286.710938,36978.507626,2415700000,15485.070312,15971.784268,53708600000,24415.0,24672.954545,0.0


In [25]:
merged.to_csv(os.path.join(OUTPUT_DIR, "unified_cpi_input.csv"), index=False)

## 7. Xử lý missing không fill target

In [ ]:
target_col = "cpi_mom"

missing_before = merged.isnull().sum().sort_values(ascending=False)

for col in merged.columns:
    if col not in ["date", target_col]:
        merged[col] = merged[col].ffill().bfill()

merged = merged.dropna(subset=[target_col]).reset_index(drop=True)

missing_after = merged.isnull().sum().sort_values(ascending=False)

print("Missing trước xử lý top 10:")
print(missing_before.head(10).to_string())
print("Missing sau xử lý:", int(merged.isnull().sum().sum()))
print("Final merged:", merged.shape)

## 8. Tạo feature modeling

In [ ]:
model_data = merged.copy()

model_data["year"] = model_data["date"].dt.year
model_data["month"] = model_data["date"].dt.month
model_data["quarter"] = model_data["date"].dt.quarter
model_data["month_sin"] = np.sin(2 * np.pi * model_data["month"] / 12)
model_data["month_cos"] = np.cos(2 * np.pi * model_data["month"] / 12)

numeric_cols = model_data.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ["year", "month", "quarter"]

# Target: chỉ tạo lag/rolling đã shift, không dùng target hiện tại làm feature
for lag in [1, 3, 6, 12]:
    model_data[f"{target_col}_lag{lag}"] = model_data[target_col].shift(lag)

for window in [3, 6, 12]:
    model_data[f"{target_col}_ma{window}_lag1"] = model_data[target_col].rolling(window, min_periods=1).mean().shift(1)

# Exogenous features
exog_cols = [c for c in numeric_cols if c not in [target_col] + exclude_cols]

for col in exog_cols:
    for lag in [1, 3, 6, 12]:
        model_data[f"{col}_lag{lag}"] = model_data[col].shift(lag)

    model_data[f"{col}_diff_lag1"] = model_data[col].diff().shift(1)

    if (model_data[col] > 0).all():
        model_data[f"{col}_dlog_lag1"] = np.log(model_data[col]).diff().shift(1)

    for window in [3, 6, 12]:
        model_data[f"{col}_ma{window}_lag1"] = model_data[col].rolling(window, min_periods=1).mean().shift(1)

model_data = model_data.dropna().reset_index(drop=True)

feature_cols = [c for c in model_data.columns if c not in ["date", target_col]]

print("Model data:", model_data.shape)
print("Số feature:", len(feature_cols))
print("Target range:")
print(model_data[["date", target_col]].tail(12).to_string(index=False))

## 9. Train/test split và lưu file

In [ ]:
split_idx = int(len(model_data) * 0.8)
train_data = model_data.iloc[:split_idx].copy()
test_data = model_data.iloc[split_idx:].copy()

full_path = os.path.join(PROCESSED_DIR, "unified_cpi_input_dataset.csv")
train_path = os.path.join(PROCESSED_DIR, "unified_cpi_train.csv")
test_path = os.path.join(PROCESSED_DIR, "unified_cpi_test.csv")
features_path = os.path.join(PROCESSED_DIR, "unified_cpi_features.txt")
columns_info_path = os.path.join(PROCESSED_DIR, "unified_cpi_columns_info.csv")
summary_path = os.path.join(PROCESSED_DIR, "unified_cpi_summary_stats.csv")

model_data.to_csv(full_path, index=False, encoding="utf-8")
train_data.to_csv(train_path, index=False, encoding="utf-8")
test_data.to_csv(test_path, index=False, encoding="utf-8")

with open(features_path, "w", encoding="utf-8") as f:
    f.write("
".join(feature_cols))

columns_info = pd.DataFrame({
    "column": model_data.columns,
    "dtype": model_data.dtypes.astype(str).values,
    "non_null_count": model_data.count().values,
    "null_count": model_data.isnull().sum().values
})
columns_info.to_csv(columns_info_path, index=False, encoding="utf-8")

model_data.describe(include="all").T.to_csv(summary_path, encoding="utf-8")

print("Full:", model_data.shape, model_data["date"].min().date(), "đến", model_data["date"].max().date())
print("Train:", train_data.shape, train_data["date"].min().date(), "đến", train_data["date"].max().date())
print("Test:", test_data.shape, test_data["date"].min().date(), "đến", test_data["date"].max().date())
print("Đã lưu:")
print(full_path)
print(train_path)
print(test_path)
print(features_path)
print(columns_info_path)
print(summary_path)

## 10. Kiểm tra nhanh target

In [ ]:
print("Target cpi_mom phải là % MoM, không phải CPI index quanh 100.")
print(model_data["cpi_mom"].describe().to_string())

print("
Test target:")
print(test_data[["date", "cpi_mom", "cpi_index"]].tail(20).to_string(index=False))